# StarCoder2-7B cross-language replication (LP4FM)

**Runtime → GPU (A100 high-RAM recommended; L4 works, slower). Run all.**

Replicates the LP4FM readout contrast on a second, larger model: the same
54-cell grid as the committed Qwen2.5-Coder-1.5B masked-probe run — 3 roles ×
6 ordered language pairs × 3 conditions (trained span-pooled, trained
context16-pooled, random-init s0 context16) — with `bigcode/starcoder2-7b`.
The masked surface baseline involves no model, so the committed
`results/lp4fm/masked_probe/surface/` files are reused unchanged; this
notebook's output is exactly 54 probe JSONs.

**Safe to pause and re-run at any point.** Every step skips completed work,
each probe JSON is verified on Drive the moment it finishes, and setup
restores everything on the next run. With `STORE_TO_DRIVE = True` a
disconnect costs at most the one in-flight probe or store pass; with the
default `False`, activation stores live only on /content, so a disconnect
after extraction re-runs extraction (~60–90 min on A100) but never loses a
finished probe. Partial or truncated copies are detected and deleted, never
resumed into: a truncated shard that were "resumed" would be silently
zero-padded by numpy, so any store that is not verifiably complete or
verifiably mid-extraction is rebuilt from scratch.

Costs, computed rather than guessed:

| step | work | time |
|---|---|---|
| extraction | ~869 shared problems × 3 langs × 3 conditions ≈ 7.8k forwards (one per program; occurrences share programs) | ~60–90 min on A100 |
| stores | 9 stores × ~2–3.5 GB (fp16, 33 layers × 4608 dims) ≈ 20–30 GB on /content | — |
| probes | 54 files, CPU layer sweeps over 4608-dim features | budget several hours; fully resumable |

Set `STORE_TO_DRIVE = True` in cell 1 only if your Drive has ~25 GB free: it
lets probe-only sessions run on a CPU runtime and makes resume after a
disconnect nearly free. Probe seeds match the Qwen protocol exactly: default
five seeds for the two trained conditions, `--seeds 0 1` for the random-init
floor.

In [ ]:
# 1 - setup
import pathlib, os, shutil, subprocess, json
BRANCH = "main"
REPO = "/content/code-model-interpretability"
STORE_TO_DRIVE = False  # 9 stores ~= 20-30 GB on Drive; see the header note
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/code-model-interpretability.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH} && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
try:
    drive.mount("/content/drive")
except Exception as e:
    print(f"retrying mount ({e})")
    drive.mount("/content/drive", force_remount=True)

MK = "/content/drive/MyDrive/code-model-interpretability/masked"
!mkdir -p data/xlcost outputs/role_occ outputs/activations_xlcost outputs/crosslang \
  {MK}/crosslang {MK}/stores {MK}/role_occ {MK}/data_xlcost

def store_state(d):
    """'complete' | 'resumable' (extraction's own partial state) | 'absent' | 'poisoned'.
    Extraction pre-sizes the shard memmap to full length, so a short shard is
    ALWAYS a truncated copy, never a legitimate in-progress store. Re-running
    extraction over one would zero-pad it silently (numpy r+ extends the file)
    and every downstream probe would run on zero vectors with no error --
    poisoned stores must be deleted, not resumed."""
    p = pathlib.Path(d)
    if not p.exists():
        return "absent"
    try:
        meta = json.load(open(p / "meta.json"))
        n0, n1, n2 = meta["shape"]
        n_occ = meta["n_occurrences"]
    except Exception:
        return "poisoned"  # missing or truncated meta.json
    shard = p / "shard.npy"
    if not shard.exists() or shard.stat().st_size < n0 * n1 * n2 * 2:
        return "poisoned"
    idx = p / "index.jsonl"
    rows = sum(1 for _ in open(idx)) if idx.exists() else 0
    if rows > n_occ:
        return "poisoned"
    return "complete" if rows == n_occ else "resumable"

def persist_file(src, dst_dir):
    """Copy with verification: a quota-full or interrupted Drive write must
    fail loudly, not print into a wall of logs and report success later."""
    src = pathlib.Path(src)
    dst = pathlib.Path(dst_dir) / src.name
    shutil.copy2(src, dst)
    if dst.stat().st_size != src.stat().st_size:
        raise SystemExit(f"Drive copy of {src.name} is truncated "
                         f"({dst.stat().st_size} vs {src.stat().st_size} bytes) -- "
                         "check Drive quota, then re-run this cell")
    return dst

def persist_store(local, drive_dir, name):
    """Stage to a temp dir, then rename: an interrupted direct copy would
    leave a partial store that a later session could restore."""
    tmp = pathlib.Path(drive_dir) / (name + ".partial")
    final = pathlib.Path(drive_dir) / name
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(local, tmp)
    if final.exists():
        shutil.rmtree(final)
    tmp.rename(final)
    if store_state(final) != "complete":
        raise SystemExit(f"store {name} did not persist to Drive intact -- check quota")

# Restore inputs and finished probe JSONs (no-clobber is safe: cell 2
# sha-verifies the inputs and cell 4 completeness-checks every JSON).
!cp -n {MK}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n {MK}/data_xlcost/* data/xlcost/ 2>/dev/null || true
!cp -n {MK}/crosslang/*starcoder27b*.json outputs/crosslang/ 2>/dev/null || true
# Stores are restored ONLY when verifiably complete on Drive, and a partial
# local copy is deleted first: cp -n cannot heal a half-copied shard.
for d in sorted(pathlib.Path(f"{MK}/stores").glob("isect_*starcoder27b*")):
    if not d.is_dir() or d.name.endswith(".partial"):
        continue
    local = pathlib.Path("outputs/activations_xlcost") / d.name
    if store_state(d) == "complete" and store_state(local) != "complete":
        if local.exists():
            shutil.rmtree(local)
        print(f"  restoring {d.name} from Drive")
        shutil.copytree(d, local)

# Existence is not the check: main carries every script below, but an older
# ref lacks --pool / --random-init / --label-field and cell 3 would die on an
# unrecognised argument. Assert the capability each cell actually uses.
REQUIRED = {
    "scripts/extract_activations.py": ["--pool", "--context-tokens", "--random-init",
                                       "--random-seed", "--label-field", "--log-every"],
    "scripts/crosslang.py":           ["--train-store", "--min-shared"],
    "scripts/tokenizer_gate.py":      ["--models"],
    "scripts/xlcost_data.py":         [],
    "scripts/role_occurrences.py":    [],
    "scripts/build_intersection.py":  [],
}
problems = []
for script, flags in REQUIRED.items():
    if not pathlib.Path(script).exists():
        problems.append(f"{script} is missing")
        continue
    if not flags:
        continue
    helps = ""
    for argv in ([script, "--help"], [script, "run", "--help"]):
        r = subprocess.run(["python"] + argv, capture_output=True, text=True)
        helps += r.stdout + r.stderr
    absent = [f for f in flags if f not in helps]
    if absent:
        problems.append(f"{script} does not accept {absent}")
if problems:
    raise SystemExit(f"BRANCH={BRANCH!r} is not the right ref:\n  " + "\n  ".join(problems))

import torch, psutil
ram = psutil.virtual_memory().total / 2**30
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"setup ok | cuda={torch.cuda.is_available()} ({gpu}) | RAM {ram:.0f} GB")
if ram < 24:
    print("WARNING: the probe cells stack two 7B stores in RAM (~9 GB) -- "
          "use a high-RAM runtime (A100/L4) or expect the kernel to die.")

In [ ]:
# 2 - config + inputs. Rebuilds guard on the .stats.json completion markers
#     the scripts write (a build killed mid-write leaves the output without
#     its marker), then the inputs are sha-verified against the committed
#     Qwen run: different inputs would make the StarCoder2 numbers
#     incomparable rather than visibly wrong.
import hashlib, itertools, re as _re
MODEL = "bigcode/starcoder2-7b"
LANGS = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT = "train"
CT = 16
ALLOW_UNVERIFIED_INPUTS = False  # see the SystemExit below before flipping this
slug = lambda mid: _re.sub(r"[^a-z0-9]", "", mid.split("/")[-1].lower())
SPAN_SLUG = slug(MODEL)                                       # -> starcoder27b
CTX_SLUG  = slug(f"{MODEL}#pool-context{CT}")                 # -> starcoder27bpoolcontext16
RCTX_SLUG = slug(f"{MODEL}#random-init-s0#pool-context{CT}")  # -> starcoder27brandominits0poolcontext16
ROLES = ("accumulator", "iterator", "index_key")
RUNS = ((SPAN_SLUG, ""), (CTX_SLUG, ""), (RCTX_SLUG, "--seeds 0 1"))
JOBS = [(role, a, b, sl, seeds) for sl, seeds in RUNS for role in ROLES
        for a, b in itertools.permutations(LANGS.values(), 2)]

def probe_complete(path):
    p = pathlib.Path(path)
    if not p.exists():
        return False
    try:
        return "transfer_macro_f1_mean" in json.load(open(p))
    except Exception:
        return False  # truncated by an interrupted write or copy

for L, s in LANGS.items():
    if not pathlib.Path(f"data/xlcost/{s}_{SPLIT}.jsonl.stats.json").exists() \
       and not pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/xlcost_data.py build --language "{L}" --split {SPLIT} --out-dir data/xlcost
for L, s in LANGS.items():
    occ = f"outputs/role_occ/all_{s}_{SPLIT}.jsonl"
    if not pathlib.Path(occ + ".stats.json").exists() \
       and not pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/role_occurrences.py extract --input data/xlcost/{s}_{SPLIT}.jsonl --role all --output {occ}
if not all(pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl.stats.json").exists()
           for s in LANGS.values()):
    !python scripts/build_intersection.py
absent = [s for s in LANGS.values()
          if not pathlib.Path(f"data/xlcost/{s}_{SPLIT}_isect.jsonl").exists()
          or not pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl").exists()]
if absent:
    raise SystemExit(f"intersection build produced nothing for {absent}; read the output above.")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

PARITY_VERIFIED = True
for L, s in LANGS.items():
    mp = pathlib.Path(f"{MK}/stores/isect_{s}_{SPLIT}_qwen25coder15b/meta.json")
    if not mp.exists():
        PARITY_VERIFIED = False
        if not ALLOW_UNVERIFIED_INPUTS:
            raise SystemExit(
                f"{s}: no Qwen store meta at {mp} -- cannot verify that these inputs "
                "are the committed run's, and unverified inputs would spend the GPU "
                "budget on incomparable numbers. Put the Qwen isect stores (or at "
                "least their meta.json files) on Drive, or set "
                "ALLOW_UNVERIFIED_INPUTS = True to proceed anyway.")
        print(f"  WARNING {s}: input parity UNVERIFIED (no Qwen store meta on Drive)")
        continue
    qm = json.load(open(mp))
    if qm.get("canonical_sha256"):
        if (qm["canonical_sha256"] != sha256(f"data/xlcost/{s}_{SPLIT}_isect.jsonl")
                or qm["occurrences_sha256"] != sha256(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl")):
            raise SystemExit(
                f"{s}: inputs differ from the committed Qwen run. Delete the local "
                f"data/xlcost and outputs/role_occ copies AND the {MK}/data_xlcost / "
                f"{MK}/role_occ files for {s} (a truncated Drive copy reproduces this "
                "error forever), then re-run from cell 1.")
        print(f"  {s}: inputs match the committed Qwen run (sha256)")
    else:
        n = sum(1 for _ in open(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl"))
        if n != qm["n_occurrences"]:
            raise SystemExit(f"{s}: {n} occurrences vs Qwen store's {qm['n_occurrences']} -- stale inputs")
        print(f"  {s}: occurrence count matches the Qwen store ({n}; legacy meta, no hashes)")

# Tokenizer offset gate: the DeepSeek incident showed corrupted offsets label
# and pool the WRONG tokens silently. Make this run self-verifying.
r = subprocess.run(["python", "scripts/tokenizer_gate.py", "run",
                    "--models", MODEL, "--output", "outputs/gate_starcoder27b.json"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-1500:])
if r.returncode != 0:
    raise SystemExit("tokenizer gate FAILED for starcoder2-7b -- do not extract")

In [ ]:
# 3 - GPU extraction, but only for stores a PENDING probe actually needs:
#     with all 54 JSONs done this cell is a no-op even on a CPU runtime.
pending = [(role, a, b, sl, seeds) for role, a, b, sl, seeds in JOBS
           if not probe_complete(f"outputs/crosslang/probe_{role}_{a}_to_{b}_{sl}.json")]
needed = sorted({(sl, s) for role, a, b, sl, _ in pending for s in (a, b)})
print(f"{len(pending)}/{len(JOBS)} probes pending -> {len(needed)} stores needed")

EXTRA = {SPAN_SLUG: "", CTX_SLUG: f"--pool context --context-tokens {CT}",
         RCTX_SLUG: f"--pool context --context-tokens {CT} --random-init --random-seed 0"}
todo = [(sl, s) for sl, s in needed
        if store_state(f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}") != "complete"]
if todo and not torch.cuda.is_available():
    raise SystemExit(f"{len(todo)} needed stores are not extracted and this runtime "
                     "has no GPU. Use a GPU runtime (or STORE_TO_DRIVE=True in a "
                     "previous GPU session to enable CPU probe-only sessions).")

for sl, s in needed:
    out = f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}"
    name = f"isect_{s}_{SPLIT}_{sl}"
    st = store_state(out)
    if st == "complete":
        print(f"  skip {name} (complete)")
    else:
        if st == "poisoned":
            print(f"  {name}: unusable partial copy -- deleting and re-extracting")
            shutil.rmtree(out)
        extra = EXTRA[sl]
        !python scripts/extract_activations.py run \
          --canonical data/xlcost/{s}_{SPLIT}_isect.jsonl \
          --occurrences outputs/role_occ/isect_{s}_{SPLIT}.jsonl \
          --model-id {MODEL} --label-field role {extra} --out-dir {out} --log-every 1000
        if store_state(out) != "complete":
            raise SystemExit(f"{out} is incomplete after extraction; see the log above")
        skips = sum(1 for l in open(f"{out}/index.jsonl") if '"skip"' in l)
        print(f"  {name}: extracted ({skips} occurrences skipped, e.g. truncation past --max-length)")
    if STORE_TO_DRIVE and store_state(f"{MK}/stores/{name}") != "complete":
        print(f"  persisting {name} to Drive")
        persist_store(out, f"{MK}/stores", name)
!du -sh outputs/activations_xlcost/isect_*starcoder27b* 2>/dev/null || true
# Persist the small inputs NOW so a reconnect never rebuilds them. Overwrite
# (with verification) only when parity was verified this session; a truncated
# Drive input must not survive, but neither should drifted rebuilds replace
# verified originals.
for pat, drive_dir in (("outputs/role_occ/*", f"{MK}/role_occ"),
                       ("data/xlcost/*", f"{MK}/data_xlcost")):
    for f in pathlib.Path().glob(pat):
        if not f.is_file():
            continue
        dst = pathlib.Path(drive_dir) / f.name
        if PARITY_VERIFIED:
            if not dst.exists() or dst.stat().st_size != f.stat().st_size:
                persist_file(f, drive_dir)
        elif not dst.exists():
            persist_file(f, drive_dir)

In [ ]:
# 4 - probes: 54 files. Each is completeness-checked, then verified on Drive
#     the moment it finishes -- and the Drive copy is re-verified even on the
#     skip path, so a copy interrupted in an earlier session is repaired.
import time
done, t0 = 0, time.time()
for role, a, b, sl, seeds in JOBS:
    out = f"outputs/crosslang/probe_{role}_{a}_to_{b}_{sl}.json"
    p = pathlib.Path(out)
    if probe_complete(p):
        dr = pathlib.Path(f"{MK}/crosslang") / p.name
        if not dr.exists() or dr.stat().st_size != p.stat().st_size:
            persist_file(p, f"{MK}/crosslang")
        done += 1
        continue
    if p.exists():
        p.unlink()  # truncated by an interrupted write: re-run, don't trust
    !python scripts/crosslang.py run \
      --train-store outputs/activations_xlcost/isect_{a}_{SPLIT}_{sl} \
      --test-store  outputs/activations_xlcost/isect_{b}_{SPLIT}_{sl} \
      --role {role} {seeds} --output {out}
    if not probe_complete(p):
        raise SystemExit(f"{out} did not complete; see the traceback above")
    persist_file(p, f"{MK}/crosslang")
    done += 1
    print(f"  [{done}/{len(JOBS)}] {role} {a}->{b} {sl} | {(time.time()-t0)/60:.0f} min elapsed")

In [ ]:
# 5 - sanity readout: point boundaries next to the committed Qwen values.
#     Intervals and CSVs are computed repo-side after the JSONs are handed off
#     (make_boundary_contrasts.py --slug-base starcoder27b).
from statistics import mean
QWEN = {"span trained": (0.852, 0.840), "context trained": (0.630, 0.569),
        "context untrained s0": (0.527, 0.464)}
for name, sl in (("span trained", SPAN_SLUG), ("context trained", CTX_SLUG),
                 ("context untrained s0", RCTX_SLUG)):
    close, far = [], []
    for role in ROLES:
        for a, b in itertools.permutations(LANGS.values(), 2):
            p = pathlib.Path(f"outputs/crosslang/probe_{role}_{a}_to_{b}_{sl}.json")
            if not probe_complete(p):
                continue
            v = json.load(open(p))["transfer_macro_f1_mean"]
            (far if "python" in (a, b) else close).append(v)
    if not (close and far):
        print(f"  {name:22s} no files yet")
        continue
    tag = "" if len(close) + len(far) == 18 else f"  (PARTIAL {len(close)+len(far)}/18 cells)"
    qc, qf = QWEN[name]
    print(f"  {name:22s} close {mean(close):.3f}  python {mean(far):.3f}  "
          f"boundary {mean(far)-mean(close):+.3f}  |  qwen {qc:.3f}/{qf:.3f}{tag}")

In [ ]:
# 6 - final verification + hand-off. The count is taken from DRIVE, parsing
#     every file: "saved" means verified there, not copied-and-hoped.
n_drive = 0
bad = []
for f in sorted(pathlib.Path(f"{MK}/crosslang").glob("probe_*starcoder27b*.json")):
    try:
        ok = "transfer_macro_f1_mean" in json.load(open(f))
    except Exception:
        ok = False
    if ok:
        n_drive += 1
    else:
        bad.append(f.name)
if bad:
    print(f"WARNING: {len(bad)} corrupt file(s) on Drive: {bad}\n"
          "re-run cell 4 to repair them from the local copies")
if STORE_TO_DRIVE:
    for sl in (SPAN_SLUG, CTX_SLUG, RCTX_SLUG):
        for s in LANGS.values():
            name = f"isect_{s}_{SPLIT}_{sl}"
            local = f"outputs/activations_xlcost/{name}"
            if store_state(local) == "complete" and store_state(f"{MK}/stores/{name}") != "complete":
                print(f"  persisting {name} to Drive")
                persist_store(local, f"{MK}/stores", name)
n_local = sum(1 for f in pathlib.Path("outputs/crosslang").glob("probe_*starcoder27b*.json")
              if probe_complete(f))
print(f"{n_drive}/54 probe JSONs verified on Drive at {MK}/crosslang/ ({n_local} local)")
if n_drive == 54:
    print("hand-off: download the 54 probe_*starcoder27b*.json files from "
          "Drive code-model-interpretability/masked/crosslang to ~/Downloads")